## 1. Import Libraries and Setup


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

## 2. Data Loading


In [2]:
# Configuration
MERGED_DATA_DIR = '../../data/merged'
FINAL_DATA_DIR = '../../data/final'

# Ensure output directory exists
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

print("=" * 80)
print("Merged Dataset Preprocessing Pipeline")
print("=" * 80)
print(f"\nMerged data location: {MERGED_DATA_DIR}")
print(f"Final location: {FINAL_DATA_DIR}\n")

Merged Dataset Preprocessing Pipeline

Merged data location: ../../data/merged
Final location: ../../data/final



In [3]:
MERGED_FILE = os.path.join(MERGED_DATA_DIR, 'merged_1.csv')
# Load merged dataset
df = pd.read_csv(MERGED_FILE)


In [4]:
# Display initial info
print("=" * 80)
print("DATAFRAME BEFORE ENCODING")
print("=" * 80)

print(f"\nShape: {df.shape}")
print(f"\nColumns ({len(df.columns)}): {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

DATAFRAME BEFORE ENCODING

Shape: (32548, 21)

Columns (21): ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'avg_clicks_per_day', 'module_presentation_length', 'tma_cma_weighted_score', 'total_submissions', 'late_submissions_count', 'avg_submission_delay', 'sites_revisit_ratio', 'activity_diversity_ratio']

Data types:
code_module                    object
code_presentation              object
id_student                      int64
gender                         object
region                         object
highest_education              object
imd_band                       object
age_band                       object
num_of_prev_attempts            int64
studied_credits                 int64
disability                     object
final_result                   object
date_registration             float64
avg_clicks_per_day    

## 3. Clean and Prepare Data

Drop id_student and region to remove PII and irrelevant features.

In [5]:
# Drop columns: PII and irrelevant features
columns_to_drop = ['id_student', 'region']
columns_to_drop = [col for col in columns_to_drop if col in df.columns]

df_encoded = df.drop(columns=columns_to_drop)

print("\n" + "=" * 80)
print("STEP 1: DATA CLEANING")
print("=" * 80)
print(f"\nColumns dropped (PII + irrelevant): {columns_to_drop}")
print(f"Shape after dropping: {df_encoded.shape}")
print(f"Remaining columns: {df_encoded.columns.tolist()}")


STEP 1: DATA CLEANING

Columns dropped (PII + irrelevant): ['id_student', 'region']
Shape after dropping: (32548, 19)
Remaining columns: ['code_module', 'code_presentation', 'gender', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'avg_clicks_per_day', 'module_presentation_length', 'tma_cma_weighted_score', 'total_submissions', 'late_submissions_count', 'avg_submission_delay', 'sites_revisit_ratio', 'activity_diversity_ratio']


## 4. Apply Ordinal Mappings

Map educational levels, age bands, and socioeconomic indices to ordinal values.


In [6]:
print("\n" + "=" * 80)
print("STEP 2: ORDINAL MAPPINGS")
print("=" * 80)

# 1. Highest Education Mapping
education_mapping = {
    'No Formal quals': 0,
    'Lower Than A Level': 1,
    'A Level or Equivalent': 2,
    'HE Qualification': 3,
    'Post Graduate Qualification': 4
}

if 'highest_education' in df_encoded.columns:
    print(f"\nBefore education mapping:")
    print(df_encoded['highest_education'].value_counts())
    
    df_encoded['highest_education'] = df_encoded['highest_education'].map(education_mapping)
    print(f"After education mapping:")
    print(df_encoded['highest_education'].value_counts(dropna=False))



STEP 2: ORDINAL MAPPINGS

Before education mapping:
highest_education
A Level or Equivalent          14026
Lower Than A Level             13138
HE Qualification                4725
No Formal quals                  346
Post Graduate Qualification      313
Name: count, dtype: int64
After education mapping:
highest_education
2    14026
1    13138
3     4725
0      346
4      313
Name: count, dtype: int64


In [7]:
# 2. Age Band Mapping
age_mapping = {
    '0-35': 0,
    '35-55': 1,
    '55+': 2
}

if 'age_band' in df_encoded.columns:
    print(f"\nBefore age_band mapping:")
    print(df_encoded['age_band'].value_counts())
    
    df_encoded['age_band'] = df_encoded['age_band'].map(age_mapping)
    print(f"After age_band mapping:")
    print(df_encoded['age_band'].value_counts(dropna=False))



Before age_band mapping:
age_band
0-35     22911
35-55     9423
55+        214
Name: count, dtype: int64
After age_band mapping:
age_band
0    22911
1     9423
2      214
Name: count, dtype: int64


In [8]:
# 3. IMD Band Mapping with region-based imputation for 'None'
imd_mapping = {
    '0-10%': 0,
    '10-20%': 1,
    '20-30%': 2,
    '30-40%': 3,
    '40-50%': 4,
    '50-60%': 5,
    '60-70%': 6,
    '70-80%': 7,
    '80-90%': 8,
    '90-100%': 9
}

if 'imd_band' in df_encoded.columns:
    print(f"\nBefore imd_band mapping:")
    print(df_encoded['imd_band'].value_counts(dropna=False))
    
    # Handle 'None' by imputing mode per region
    none_mask = df_encoded['imd_band'] == 'None'
    
    if none_mask.sum() > 0 and 'region' in df_encoded.columns:
        print(f"\nImputing {none_mask.sum()} 'None' values by region mode...")
        
        for region in df_encoded.loc[none_mask, 'region'].unique():
            region_mask = (df_encoded['region'] == region) & ~none_mask
            if region_mask.sum() > 0:
                mode_value = df_encoded.loc[region_mask, 'imd_band'].mode()
                if len(mode_value) > 0:
                    df_encoded.loc[(df_encoded['region'] == region) & none_mask, 'imd_band'] = mode_value[0]
    
    # Apply mapping
    df_encoded['imd_band'] = df_encoded['imd_band'].map(imd_mapping)
    print(f"After imd_band mapping:")
    print(df_encoded['imd_band'].value_counts(dropna=False))


Before imd_band mapping:
imd_band
10-20%     4242
20-30%     3651
0-10%      3618
30-40%     3541
40-50%     3250
50-60%     3130
60-70%     2899
70-80%     2877
80-90%     2758
90-100%    2582
Name: count, dtype: int64
After imd_band mapping:
imd_band
1    4242
2    3651
0    3618
3    3541
4    3250
5    3130
6    2899
7    2877
8    2758
9    2582
Name: count, dtype: int64


## 5. Perform One-Hot Encoding

Use pd.get_dummies for categorical features: code_module, code_presentation, and gender. Drop first category to avoid multicollinearity.


In [9]:
print("\n" + "=" * 80)
print("STEP 3: ONE-HOT ENCODING")
print("=" * 80)

# Identify categorical columns for one-hot encoding
categorical_cols = ['code_module', 'code_presentation', 'gender', 'disability']
categorical_cols = [col for col in categorical_cols if col in df_encoded.columns]

print(f"\nColumns to one-hot encode: {categorical_cols}")

# Apply one-hot encoding (drop first category to avoid multicollinearity)
df_encoded = pd.get_dummies(
    df_encoded,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)



STEP 3: ONE-HOT ENCODING

Columns to one-hot encode: ['code_module', 'code_presentation', 'gender', 'disability']


In [10]:

print(f"\nShape after one-hot encoding: {df_encoded.shape}")
print(f"\nNew columns created:")
new_cols = [col for col in df_encoded.columns if any(cat in col for cat in categorical_cols)]
print(new_cols)

print(f"\nTotal columns now: {len(df_encoded.columns)}")


Shape after one-hot encoding: (32548, 26)

New columns created:
['code_module_BBB', 'code_module_CCC', 'code_module_DDD', 'code_module_EEE', 'code_module_FFF', 'code_module_GGG', 'code_presentation_2013J', 'code_presentation_2014B', 'code_presentation_2014J', 'gender_M', 'disability_Y']

Total columns now: 26


## 5a. Feature Removal: Statistical Justification

Remove features with minimal contribution to behavioral clustering:
- **avg_submission_delay**: PCA loading of 0.99 on PC1 dominates variance, masking other behavioral patterns
- **Demographics** (gender_M, disability_Y, age_band, imd_band, highest_education): MI scores < 0.02, low PCA loadings (mathematical noise)
- **Presentation Metadata** (code_presentation_*, module_presentation_length): Insufficient MI scores, not behavioral indicators

In [11]:
print("\n" + "=" * 80)
print("STEP 3a: FEATURE REMOVAL - STATISTICAL JUSTIFICATION")
print("=" * 80)

print(f"\nShape before feature removal: {df_encoded.shape}")
print(f"Columns before: {len(df_encoded.columns)}")

# Define features to remove with justification
features_to_remove = {
    'Variance Dominance': {
        'features': ['avg_submission_delay'],
        'reason': 'PCA loading 0.99 on PC1 dominates variance, masks other behavioral patterns'
    },
    'Demographic Noise (MI < 0.02, low PCA loadings)': {
        'features': ['gender_M', 'disability_Y', 'age_band', 'imd_band', 'highest_education'],
        'reason': 'Near-zero mutual information, act as mathematical noise in clustering'
    },
    'Presentation Metadata (insufficient MI)': {
        'features': [col for col in df_encoded.columns if col.startswith('code_presentation_')] + ['module_presentation_length'],
        'reason': 'Not behavioral indicators; insufficient MI scores for inclusion'
    }
}

# Collect all features to remove
all_features_to_remove = []
for category, info in features_to_remove.items():
    features = [f for f in info['features'] if f in df_encoded.columns]
    if features:
        print(f"\n{category}:")
        print(f"  Reason: {info['reason']}")
        print(f"  Features to remove: {features}")
        all_features_to_remove.extend(features)

# Remove duplicate entries
all_features_to_remove = list(set(all_features_to_remove))

# Drop the features
df_encoded = df_encoded.drop(columns=all_features_to_remove)

print(f"\n" + "-" * 80)
print(f"✓ Features removed: {len(all_features_to_remove)}")
print(f"  Removed: {sorted(all_features_to_remove)}")
print(f"\nShape after feature removal: {df_encoded.shape}")
print(f"Columns after: {len(df_encoded.columns)}")
print(f"\nRemaining features (pure behavioral):")
print(f"  {sorted(df_encoded.columns.tolist())}")


STEP 3a: FEATURE REMOVAL - STATISTICAL JUSTIFICATION

Shape before feature removal: (32548, 26)
Columns before: 26

Variance Dominance:
  Reason: PCA loading 0.99 on PC1 dominates variance, masks other behavioral patterns
  Features to remove: ['avg_submission_delay']

Demographic Noise (MI < 0.02, low PCA loadings):
  Reason: Near-zero mutual information, act as mathematical noise in clustering
  Features to remove: ['gender_M', 'disability_Y', 'age_band', 'imd_band', 'highest_education']

Presentation Metadata (insufficient MI):
  Reason: Not behavioral indicators; insufficient MI scores for inclusion
  Features to remove: ['code_presentation_2013J', 'code_presentation_2014B', 'code_presentation_2014J', 'module_presentation_length']

--------------------------------------------------------------------------------
✓ Features removed: 10
  Removed: ['age_band', 'avg_submission_delay', 'code_presentation_2013J', 'code_presentation_2014B', 'code_presentation_2014J', 'disability_Y', 'gen

## 6. Scale Numerical Features

Scaling strategy based on feature characteristics:
- **RobustScaler (High Outliers)**: avg_clicks_per_day, studied_credits, sites_revisit_ratio, activity_diversity_ratio, avg_submission_delay, total_submissions
- **MinMaxScaler (Bounded)**: tma_cma_weighted_score, highest_education, imd_band, age_band
- **Log Transform + StandardScaler (Skewed/Count)**: late_submissions_count, num_of_prev_attempts
- **StandardScaler (Normal)**: date_registration, module_presentation_length

In [12]:
print("\n" + "=" * 80)
print("STEP 4: NUMERICAL FEATURE SCALING")
print("=" * 80)

# Define feature groups with different scaling strategies
# Based on feature characteristics guide
features_robust_outliers = ['avg_clicks_per_day', 'studied_credits', 'sites_revisit_ratio', 
                            'activity_diversity_ratio', 'avg_submission_delay', 'total_submissions']
features_minmax_bounded = ['tma_cma_weighted_score', 'highest_education', 'imd_band', 'age_band']
features_log_standard = ['late_submissions_count', 'num_of_prev_attempts']
features_standard_normal = ['date_registration', 'module_presentation_length']

# Check which features exist in the dataframe
robust_present = [col for col in features_robust_outliers if col in df_encoded.columns]
minmax_present = [col for col in features_minmax_bounded if col in df_encoded.columns]
log_standard_present = [col for col in features_log_standard if col in df_encoded.columns]
standard_present = [col for col in features_standard_normal if col in df_encoded.columns]

print(f"\n--- Feature Scaling Strategy ---")
print(f"RobustScaler (High Outliers): {robust_present}")
print(f"MinMaxScaler (Bounded): {minmax_present}")
print(f"Log + StandardScaler (Skewed/Count): {log_standard_present}")
print(f"StandardScaler (Normal): {standard_present}")

# 1. Apply RobustScaler to high outlier features
if robust_present:
    print(f"\n--- RobustScaler (High Outliers) ---")
    print(f"Before scaling:")
    print(df_encoded[robust_present].describe())
    
    scaler_robust = RobustScaler()
    df_encoded[robust_present] = scaler_robust.fit_transform(df_encoded[robust_present])
    
    print(f"After scaling:")
    print(df_encoded[robust_present].describe())

# 2. Apply MinMaxScaler to bounded features (including ordinal)
if minmax_present:
    print(f"\n--- MinMaxScaler (Bounded) ---")
    print(f"Before scaling:")
    print(df_encoded[minmax_present].describe())
    
    scaler_minmax = MinMaxScaler()
    df_encoded[minmax_present] = scaler_minmax.fit_transform(df_encoded[minmax_present])
    
    print(f"After scaling (should be in [0, 1]):")
    print(df_encoded[minmax_present].describe())

# 3. Apply Log Transform + StandardScaler to skewed count features
if log_standard_present:
    print(f"\n--- Log Transform + StandardScaler (Skewed/Count) ---")
    print(f"Before scaling:")
    print(df_encoded[log_standard_present].describe())
    
    # Apply log1p transform first (handles zeros)
    df_encoded[log_standard_present] = np.log1p(df_encoded[log_standard_present])
    print(f"After log1p transform:")
    print(df_encoded[log_standard_present].describe())
    
    # Then apply StandardScaler
    scaler_standard_log = StandardScaler()
    df_encoded[log_standard_present] = scaler_standard_log.fit_transform(df_encoded[log_standard_present])
    
    print(f"After StandardScaler:")
    print(df_encoded[log_standard_present].describe())

# 4. Apply StandardScaler to normal features
if standard_present:
    print(f"\n--- StandardScaler (Normal) ---")
    print(f"Before scaling:")
    print(df_encoded[standard_present].describe())
    
    scaler_standard = StandardScaler()
    df_encoded[standard_present] = scaler_standard.fit_transform(df_encoded[standard_present])
    
    print(f"After scaling:")
    print(df_encoded[standard_present].describe())

if not (robust_present or minmax_present or log_standard_present or standard_present):
    print("\nWarning: No features found to scale")


STEP 4: NUMERICAL FEATURE SCALING

--- Feature Scaling Strategy ---
RobustScaler (High Outliers): ['avg_clicks_per_day', 'studied_credits', 'sites_revisit_ratio', 'activity_diversity_ratio', 'total_submissions']
MinMaxScaler (Bounded): ['tma_cma_weighted_score']
Log + StandardScaler (Skewed/Count): ['late_submissions_count', 'num_of_prev_attempts']
StandardScaler (Normal): ['date_registration']

--- RobustScaler (High Outliers) ---
Before scaling:
       avg_clicks_per_day  studied_credits  sites_revisit_ratio  \
count        32548.000000     32548.000000         32548.000000   
mean             3.744569        79.714422             3.388834   
std              2.120344        41.046179             2.276461   
min              0.000000        30.000000             0.000000   
25%              2.534611        60.000000             2.017857   
50%              3.690916        60.000000             3.159391   
75%              5.020506       120.000000             4.407744   
max        

In [13]:
df_encoded.head()

,num_of_prev_attempts,studied_credits,final_result,date_registration,avg_clicks_per_day,tma_cma_weighted_score,total_submissions,late_submissions_count,sites_revisit_ratio,activity_diversity_ratio,code_module_BBB,code_module_CCC,code_module_DDD,code_module_EEE,code_module_FFF,code_module_GGG
0,-0.368196,3.0,Pass,-1.818699,0.765076,0.824,0.000,-0.828366,-0.051478,0.370037,0,0,0,0,0,0
1,-0.368196,0.0,Pass,0.333158,0.132223,0.654,0.000,0.651325,0.456343,0.925370,0,0,0,0,0,0
2,-0.368196,0.0,Withdrawn,-0.458563,0.309507,0.000,-0.625,-0.828366,-0.123753,-0.261894,0,0,0,0,0,0
3,-0.368196,0.0,Pass,0.353459,0.105180,0.763,0.000,-0.828366,1.464147,0.887071,0,0,0,0,0,0
4,-0.368196,0.0,Pass,-2.163809,-0.147294,0.550,0.000,1.584907,0.649709,0.580680,0,0,0,0,0,0


## 7. Export Final Encoded Dataset

Save the final encoded and scaled dataframe to data/final/encoded_unsupervised_dbscan.csv.

Keeping the final_result (target valriable), as it is needed for crosstab in dbscan

In [14]:
print("\n" + "=" * 80)
print("STEP 6: EXPORT FINAL ENCODED DATASET")
print("=" * 80)

# Ensure output directory exists
os.makedirs(FINAL_DATA_DIR, exist_ok=True)

# Save to data/final/
FINAL_FILE_PATH = os.path.join(FINAL_DATA_DIR, 'encoded_unsupervised_dbscan.csv')
df_encoded.to_csv(FINAL_FILE_PATH, index=False)

print(f"\n✓ Final encoded dataset saved to: {FINAL_FILE_PATH}")

print(f"\nFinal dataframe information:")
print(f"  Shape: {df_encoded.shape}")
print(f"  Columns: {len(df_encoded.columns)}")
print(f"  Data types:\n{df_encoded.dtypes}")
print(f"\nMissing values:")
print(df_encoded.isnull().sum())

print(f"\nFirst few rows:")
print(df_encoded.head())

print("\n" + "=" * 80)
print("ENCODING PIPELINE COMPLETE")
print("=" * 80)


STEP 6: EXPORT FINAL ENCODED DATASET

✓ Final encoded dataset saved to: ../../data/final\encoded_unsupervised_dbscan.csv

Final dataframe information:
  Shape: (32548, 16)
  Columns: 16
  Data types:
num_of_prev_attempts        float64
studied_credits             float64
final_result                 object
date_registration           float64
avg_clicks_per_day          float64
tma_cma_weighted_score      float64
total_submissions           float64
late_submissions_count      float64
sites_revisit_ratio         float64
activity_diversity_ratio    float64
code_module_BBB               int64
code_module_CCC               int64
code_module_DDD               int64
code_module_EEE               int64
code_module_FFF               int64
code_module_GGG               int64
dtype: object

Missing values:
num_of_prev_attempts        0
studied_credits             0
final_result                0
date_registration           0
avg_clicks_per_day          0
tma_cma_weighted_score      0
total_submis